# 🏠 DOMIAN — Entrenamiento Local (Mac → datos en RPi 5)
## Adaptive Classifier · 3 clases · CPU · sin GPU

**Ejecuta en la Mac, pero lee los datos y escribe el modelo en la Raspberry Pi 5.**

**Clases:**
- `absent` → habitación vacía
- `present_still` → persona quieta
- `present_moving` → persona moviéndose

**Requisitos:**
```bash
pip install numpy scikit-learn paramiko scp
```

**Importante:** las grabaciones permanecen en la RPi (`~/Documents/RuView/v2/data/recordings/`)
con prefijos `train_absent_`, `train_present_still_`, `train_present_moving_`.
Este notebook las descarga temporalmente, entrena localmente en la Mac (rápido, CPU),
y sube el modelo resultante de vuelta a la RPi.

## Paso 1 — Instalar dependencias

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'numpy', 'scikit-learn', 'paramiko', 'scp', '-q'])
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import json, os, tempfile
from pathlib import Path
import paramiko
from scp import SCPClient
print('✅ Dependencias instaladas')
print(f'   numpy {np.__version__}')


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✅ Dependencias instaladas
   numpy 2.1.3


## Paso 2 — Configurar conexión a la Raspberry Pi 5

In [ ]:
# ══════════════════════════════════════════════
# CONFIGURACIÓN — edita según tu red
# ══════════════════════════════════════════════
RPI_HOST = "192.168.0.19"
RPI_USER = "domian"
RPI_PASSWORD = "Domian123!"   # deja None y usa key-based auth, o pon tu contraseña aquí
RPI_KEY_PATH = os.path.expanduser("~/.ssh/id_rsa")  # si usas clave SSH

# Rutas remotas en la RPi
RPI_RECORDINGS_DIR = "/home/domian/Documents/RuView/v2/data/recordings"
RPI_MODEL_PATH      = "/home/domian/Documents/RuView/v2/data/adaptive_model.json"

# Directorio temporal local (Mac) para trabajar
LOCAL_TMP_DIR = tempfile.mkdtemp(prefix="domian_train_")
print(f"Directorio temporal local: {LOCAL_TMP_DIR}")

CLASE_NOMBRES = ['absent', 'present_still', 'present_moving']
LABEL_MAP = {
    'train_absent':         0,
    'train_present_still':  1,
    'train_present_moving': 2,
}

print(f"\nRPi destino: {RPI_USER}@{RPI_HOST}")
print(f"Grabaciones remotas: {RPI_RECORDINGS_DIR}")
print(f"Modelo remoto: {RPI_MODEL_PATH}")

SyntaxError: unterminated string literal (detected at line 6) (2349708347.py, line 6)

## Paso 3 — Conectar por SSH y listar grabaciones disponibles

In [ ]:
def conectar_ssh():
    """Crea cliente SSH a la RPi — usa clave si existe, sino contraseña."""
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    if RPI_PASSWORD:
        ssh.connect(RPI_HOST, username=RPI_USER, password=RPI_PASSWORD)
    elif os.path.exists(RPI_KEY_PATH):
        ssh.connect(RPI_HOST, username=RPI_USER, key_filename=RPI_KEY_PATH)
    else:
        # Fallback: pedir contraseña interactivamente
        import getpass
        pwd = getpass.getpass(f"Contraseña SSH para {RPI_USER}@{RPI_HOST}: ")
        ssh.connect(RPI_HOST, username=RPI_USER, password=pwd)
    return ssh

ssh = conectar_ssh()
print(f"✅ Conectado a {RPI_HOST}")

stdin, stdout, stderr = ssh.exec_command(f"ls -la {RPI_RECORDINGS_DIR}")
salida = stdout.read().decode()
print(salida)

# Listar solo archivos train_*
stdin, stdout, stderr = ssh.exec_command(f"ls {RPI_RECORDINGS_DIR} | grep '^train_'")
archivos_remotos = [l.strip() for l in stdout.read().decode().splitlines() if l.strip()]

print(f"\nArchivos de entrenamiento encontrados: {len(archivos_remotos)}")
for f in archivos_remotos:
    label = next((LABEL_MAP[p] for p in LABEL_MAP if f.startswith(p)), None)
    clase = CLASE_NOMBRES[label] if label is not None else '⚠️ sin prefijo válido'
    print(f"  {f} → {clase}")

## Paso 4 — Descargar grabaciones desde la RPi a la Mac (temporal)

In [ ]:
with SCPClient(ssh.get_transport()) as scp:
    for archivo in archivos_remotos:
        remote_path = f"{RPI_RECORDINGS_DIR}/{archivo}"
        local_path  = os.path.join(LOCAL_TMP_DIR, archivo)
        print(f"Descargando {archivo}...", end=" ")
        scp.get(remote_path, local_path)
        size_mb = os.path.getsize(local_path) / 1024 / 1024
        print(f"✅ {size_mb:.1f} MB")

print(f"\n✅ {len(archivos_remotos)} archivos descargados a {LOCAL_TMP_DIR}")

## Paso 5 — Cargar datos (features 15-dim, compatibles con el servidor)

In [ ]:
def extraer_features_15(obj):
    """Extrae 15 features por nodo — compatible con N_FEATURES=15 del servidor RuView"""
    resultados = []
    for nf in obj.get('node_features', []):
        f   = nf.get('features', {})
        clf = nf.get('classification', {})
        row = [
            float(f.get('mean_rssi', 0)),
            float(f.get('variance', 0)),
            float(f.get('motion_band_power', 0)),
            float(f.get('breathing_band_power', 0)),
            float(f.get('dominant_freq_hz', 0)),
            float(f.get('change_points', 0)),
            float(f.get('spectral_power', 0)),
            float(nf.get('rssi_dbm', 0)),
            float(nf.get('last_seen_ms', 0)),
            float(nf.get('frame_rate_hz', 0)),
            1.0 if nf.get('stale') else 0.0,
            1.0 if clf.get('presence') else 0.0,
            float(clf.get('confidence', 0)),
            float(nf.get('node_id', 0)),
            0.0  # padding
        ]
        resultados.append(row)
    return resultados

def cargar_grabacion(path, label):
    frames = []
    with open(path, 'r') as f:
        for linea in f:
            try:
                obj = json.loads(linea.strip())
                for row in extraer_features_15(obj):
                    frames.append((np.array(row, dtype=np.float32), label))
            except:
                continue
    return frames

datos = []
print('Cargando grabaciones descargadas...')
for archivo in archivos_remotos:
    label = None
    for prefijo, lbl in LABEL_MAP.items():
        if archivo.startswith(prefijo):
            label = lbl
            break
    if label is None:
        continue
    path   = os.path.join(LOCAL_TMP_DIR, archivo)
    frames = cargar_grabacion(path, label)
    print(f'  ✅ {archivo}: {len(frames):,} frames → {CLASE_NOMBRES[label]}')
    datos += frames

if not datos:
    print('\n❌ Sin datos válidos')
else:
    print(f'\nTotal: {len(datos):,} frames')
    for i, nombre in enumerate(CLASE_NOMBRES):
        count = sum(1 for _, l in datos if l == i)
        pct   = 100 * count / len(datos)
        print(f'  {nombre:20}: {count:,} ({pct:.1f}%)')

## Paso 6 — Entrenar con Logistic Regression (local, CPU, rápido)

In [ ]:
import time

X = np.array([f for f, _ in datos])
y = np.array([l for _, l in datos])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_n = scaler.fit_transform(X_train)
X_val_n   = scaler.transform(X_val)

mean15 = scaler.mean_
std15  = scaler.scale_

print(f'Train: {len(X_train):,} frames')
print(f'Val:   {len(X_val):,} frames')
print('\nEntrenando...')

t0  = time.time()
clf = LogisticRegression(
    max_iter=1000, C=1.0,
    multi_class='multinomial', solver='lbfgs',
    random_state=42, n_jobs=-1
)
clf.fit(X_train_n, y_train)
elapsed = time.time() - t0

acc = clf.score(X_val_n, y_val)
print(f'\n✅ Entrenamiento completado en {elapsed:.1f}s')
print(f'   Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print()
print(classification_report(y_val, clf.predict(X_val_n), target_names=CLASE_NOMBRES))

## Paso 7 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_val, clf.predict(X_val_n))

print('Confusion Matrix (filas=real, columnas=predicho):')
print(f'{"": <20}', end='')
for c in CLASE_NOMBRES:
    print(f'{c[:12]:>14}', end='')
print()
print('-' * 62)
for i, row in enumerate(cm):
    print(f'{CLASE_NOMBRES[i]:20}', end='')
    for val in row:
        print(f'{val:>14,}', end='')
    print()

print()
print('Precisión por clase:')
for i, nombre in enumerate(CLASE_NOMBRES):
    correctos = cm[i][i]
    total     = cm[i].sum()
    print(f'  {nombre:20}: {correctos:,}/{total:,} = {100*correctos/total:.1f}%')

## Paso 8 — Exportar modelo en formato RuView

In [ ]:
weights = []
for i in range(len(CLASE_NOMBRES)):
    row = clf.coef_[i].tolist() + [float(clf.intercept_[i])]
    weights.append(row)

print(f'Weights: {len(weights)} clases x {len(weights[0])} (features + bias)')

class_stats = []
for label_id, nombre in enumerate(CLASE_NOMBRES):
    mask  = y == label_id
    X_cls = X[mask]
    if len(X_cls) == 0:
        class_stats.append({'label': nombre, 'count': 0, 'mean': [0]*15, 'stddev': [1]*15})
        continue
    class_stats.append({
        'label':  nombre,
        'count':  int(mask.sum()),
        'mean':   X_cls.mean(axis=0).tolist(),
        'stddev': X_cls.std(axis=0).tolist()
    })
    print(f'  {nombre:20}: {mask.sum():,} frames')

modelo_ruview = {
    'class_stats':       class_stats,
    'weights':           weights,
    'global_mean':       mean15.tolist(),
    'global_std':        std15.tolist(),
    'trained_frames':    len(X),
    'training_accuracy': float(acc),
    'version':           1,
    'class_names':       CLASE_NOMBRES
}

local_model_path = os.path.join(LOCAL_TMP_DIR, 'adaptive_model.json')
with open(local_model_path, 'w') as f:
    json.dump(modelo_ruview, f, indent=2)

print(f'\n✅ Modelo generado localmente: {local_model_path}')
print(f'   Accuracy: {acc:.4f} ({acc*100:.1f}%)')

## Paso 9 — Subir el modelo de vuelta a la RPi

In [ ]:
with SCPClient(ssh.get_transport()) as scp:
    scp.put(local_model_path, RPI_MODEL_PATH)

print(f'✅ Modelo subido a la RPi: {RPI_MODEL_PATH}')

# Verificar que el archivo remoto tiene el contenido correcto
stdin, stdout, stderr = ssh.exec_command(
    f"python3 -c \"import json; m=json.load(open('{RPI_MODEL_PATH}')); "
    f"print('frames:', m['trained_frames']); print('accuracy:', m['training_accuracy'])\""
)
print(stdout.read().decode())
print(stderr.read().decode())

## Paso 10 — Reiniciar el servicio domian en la RPi para cargar el nuevo modelo

In [ ]:
respuesta = input('¿Reiniciar el servicio domian.service ahora? (s/n): ')

if respuesta.lower() == 's':
    stdin, stdout, stderr = ssh.exec_command('sudo systemctl restart domian')
    exit_status = stdout.channel.recv_exit_status()
    if exit_status == 0:
        print('✅ Servicio reiniciado')
    else:
        print(f'⚠️  Posible error: {stderr.read().decode()}')

    import time
    time.sleep(5)
    stdin, stdout, stderr = ssh.exec_command(
        "journalctl -u domian -n 10 --no-pager | grep -i 'adaptive classifier'"
    )
    print(stdout.read().decode())
else:
    print('Reinicia manualmente con:')
    print(f'  ssh {RPI_USER}@{RPI_HOST} "sudo systemctl restart domian"')

## Paso 11 — Limpieza y cierre

In [ ]:
import shutil

ssh.close()
print('✅ Conexión SSH cerrada')

respuesta = input(f'¿Borrar archivos temporales en {LOCAL_TMP_DIR}? (s/n): ')
if respuesta.lower() == 's':
    shutil.rmtree(LOCAL_TMP_DIR)
    print('✅ Archivos temporales eliminados')
else:
    print(f'Archivos conservados en: {LOCAL_TMP_DIR}')

print(f'\n🎉 Entrenamiento completo — accuracy {acc*100:.1f}% activo en la RPi')